# 18장 전이학습 — 남이 학습한 것 위에서 시작하기

**2단계 · 주석 달기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch18_2_comment.ipynb)

이 노트북에는 **주석을 지운 코드**가 들어 있습니다. 아래 표의 「주석 달기」 과제대로 내 말로 주석을 달아 보세요. 주석을 다는 동안 코드를 한 줄씩 다시 읽게 됩니다.

- 코드 셀 첫 줄의 `%%writefile 파일이름` 은 "이 셀의 내용을 그 파일로 저장하라"는 뜻입니다. 셀을 실행하면 내가 고친 내용이 파일에 저장됩니다.
- 그 아래 확인 셀을 실행해, 주석을 단 뒤에도 코드가 전과 똑같이 도는지 봅니다.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | `p01`, `p02` 를 치고 실행. `p01` 의 `named_children()` 출력을 17장 코드와 대조 | [ch18_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch18_1_follow.ipynb) |
| 2. 주석 달기 | `p02` 의 A·B·C 각각에 "학습되는 파라미터 / 얼린 파라미터 / 학습률"을 주석으로 | [ch18_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch18_2_comment.ipynb) **(지금 이 노트북)** |
| 3. 다시 만들기 | 14장 패키지에 `--model resnet18 --pretrained --freeze` 인자를 추가해 B 방식이 되게 하기 | [ch18_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch18_3_build.ipynb) |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "18"))                      # 이 장의 코드 폴더로 이동
if not os.path.isdir(os.path.join(os.environ["JUCOS_DATA"], "shapes")):   # 15장의 도형 데이터가 없으면 만든다
    !python ../15/make_shapes.py
print("준비 끝. 작업 폴더: code/18")

A·B·C 각각에 「학습되는 파라미터 / 얼린 파라미터 / 학습률」을 주석으로 다세요.

## `p02_finetune.py` — 대상: 파일 전체

In [ ]:
%%writefile p02_finetune.py
import os
import sys
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models, transforms

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "15"))
from p02_custom_dataset import ImageFolderDataset

ROOT = os.path.join(os.environ.get("JUCOS_DATA", "./data"), "shapes")
tf = transforms.Compose([transforms.Resize((112, 112)), transforms.ToTensor(),
                         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
train_set = ImageFolderDataset(os.path.join(ROOT, "train"), tf)
test_set = ImageFolderDataset(os.path.join(ROOT, "test"), tf)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64)
NUM_CLASSES = len(train_set.classes)


def accuracy(model):
    model.eval()
    c = 0
    with torch.no_grad():
        for x, y in test_loader:
            c += (model(x).argmax(1) == y).sum().item()
    model.train()
    return c / len(test_set)


def make_model(pretrained):
    m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    return m


def run(name, model, params, lr, epochs=2):
    torch.manual_seed(0)
    opt = torch.optim.Adam(params, lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    t0 = time.time()
    for _ in range(epochs):
        for x, y in train_loader:
            opt.zero_grad()
            loss_fn(model(x), y).backward()
            opt.step()
    n_train = sum(p.numel() for p in params)
    print(f"{name:<22} 학습 파라미터 {n_train:>10,}  정확도 {accuracy(model):.3f}  ({time.time() - t0:.0f}초)")


if __name__ == "__main__":
    torch.manual_seed(0)
    m = make_model(pretrained=False)
    run("A. 처음부터", m, list(m.parameters()), lr=1e-3)
    torch.manual_seed(0)
    m = make_model(pretrained=True)
    for p in m.parameters():
        p.requires_grad = False
    for p in m.fc.parameters():
        p.requires_grad = True
    run("B. 특징 추출(fc만)", m, list(m.fc.parameters()), lr=1e-3)
    torch.manual_seed(0)
    m = make_model(pretrained=True)
    run("C. 미세조정(전부)", m, list(m.parameters()), lr=1e-4)

### 확인

In [ ]:
!python p02_finetune.py

### 원래대로 되돌리기

파일을 책의 원래 코드로 되돌리고 싶으면 아래 셀을 실행하세요. (Colab 에서 받은 저장소 기준입니다.)

In [ ]:
!git checkout -- p02_finetune.py